# Quantum Segment-Level Analysis — store-backed (statevector)

Reads the decoupled **`qtrk_store`** via `qtrk_view` (metrics recomputed as a VIEW with the fixed, gamma-aware absolute threshold tau = delta/(delta+gamma)+0.10 = 0.35 at gamma=3 — never the old relative tau*max, never the baked pkl metrics).

**Scope:** statevector-only campaign. The old notebook's downstream-tracking figure and its statevector-vs-sampling/hires comparison suite (Sec 7e-7q) are intentionally omitted — the store has no `get_tracks` and no sampling readout. Figures fill in as the Condor quantum jobs drain.

In [ ]:
# Setup
import sys, pathlib
sys.path.insert(0, '/data/bfys/gscriven/Quantum_Track_Reconstruction/Toy_Characterisation/_shared')
sys.path.insert(0, '/data/bfys/gscriven/Quantum_Track_Reconstruction/Toy_Characterisation/Verify_new_results')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import qtrk_view as V

STUDY = 'Verify_new_results'
OUT = pathlib.Path('outputs/quantum_segment_analysis/store_backed'); OUT.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({'font.size':12,'axes.grid':True,'grid.alpha':0.3,'lines.markersize':7,'lines.linewidth':2})
cC, cQ, cT, cF = '#444444', '#2166ac', '#1b7837', '#c51b7d'

def _err(ax, a, col, fmt='o-', color=None, label=None, mfc=None):
    if a is None or len(a)==0 or f'{col}_mean' not in a.columns: return
    ax.errorbar(a['n_trk'], a[f'{col}_mean'], yerr=a[f'{col}_sem'], fmt=fmt, color=color,
                capsize=4, mfc=mfc, markeredgecolor='black', markeredgewidth=0.7, label=label, zorder=3)
def _save(fig, stem):
    fig.savefig(OUT/f'{stem}.pdf', bbox_inches='tight')
    fig.savefig(OUT/f'{stem}.png', dpi=150, bbox_inches='tight'); print('saved', stem)
def _aggcols(df, cols):
    g=df.groupby('n_trk'); out=pd.DataFrame({'n_trk':sorted(df.n_trk.unique())}).set_index('n_trk')
    for c in cols:
        out[f'{c}_mean']=g[c].mean(); out[f'{c}_sem']=g[c].apply(lambda x: x.std(ddof=1)/np.sqrt(len(x)) if len(x)>1 else 0.0)
    return out.reset_index()
print('tau =', V.threshold())

In [ ]:
# Load the store view (statevector); pair classical+quantum; raw vectors for diagnostics
view = V.load_view(study=STUDY)              # tidy fixed-metric view
m    = V.paired(view)                          # one row per (event,ham): classical + quantum
agg  = V.aggregate(m)                          # per-T mean/sem (eff/pur/far/cos for C and Q)
recs = V.vector_records(view)                  # raw sol_C/sol_Q_scaled/truth per event
print(f'paired quantum points: {len(m)}  | T = {sorted(m.n_trk.unique()) if len(m) else []}  | reps total = {len(recs)}')
if len(agg):
    print(agg[['n_trk','n_reps','eff_C_mean','eff_Q_mean','pur_Q_mean','far_Q_mean','cos_mean']].to_string(index=False))
else:
    print('(no quantum solves in the store for this study yet — figures will be empty until Condor drains)')

In [ ]:
# Fig 1 — segment metrics vs T (classical vs quantum statevector), fixed metrics
fig, ax = plt.subplots(2,2, figsize=(12,9))
_err(ax[0,0],agg,'eff_C','o--',cC,'classical',mfc='white'); _err(ax[0,0],agg,'eff_Q','o-',cQ,'quantum (sv)')
ax[0,0].axhline(1,color='gray',ls='--',lw=1); ax[0,0].set_ylim(0,1.05); ax[0,0].set_title('i) Segment efficiency'); ax[0,0].legend(loc='lower left')
_err(ax[0,1],agg,'pur_C','o--',cC,'classical',mfc='white'); _err(ax[0,1],agg,'pur_Q','o-',cQ,'quantum (sv)')
ax[0,1].set_ylim(0,1.05); ax[0,1].set_title('ii) Segment purity'); ax[0,1].legend(loc='lower left')
_err(ax[1,0],agg,'far_C','o--',cC,'classical',mfc='white'); _err(ax[1,0],agg,'far_Q','o-',cQ,'quantum (sv)')
ax[1,0].set_yscale('log'); ax[1,0].set_title('iii) False rate = N_false_act / N_active'); ax[1,0].legend(loc='upper left')
_err(ax[1,1],agg,'cos','o-',cQ,'cos(s_Q, s_C)')
ax[1,1].set_ylim(0,1.05); ax[1,1].set_title('iv) Solver cosine fidelity'); ax[1,1].legend(loc='lower left')
for a in ax.flat: a.set_xlabel('Number of tracks'); a.set_xscale('log')
fig.tight_layout(); _save(fig,'fig1_segment_metrics'); plt.show()

In [ ]:
# Fig 2 — solution-vector fidelity (cosine, Jaccard of active sets, relative L2) vs T
df = V.records_df(view)
fig, ax = plt.subplots(1,3, figsize=(15,4))
if len(df):
    fid = _aggcols(df, ['cosine','jaccard','rel_l2'])
    for j,(c,t) in enumerate([('cosine','cos(s_Q, s_C)'),('jaccard','Jaccard(A_Q, A_C)'),('rel_l2','rel. L2  ||s_Q - s_C|| / ||s_C||')]):
        _err(ax[j],fid,c,'o-',cQ); ax[j].set_title(t); ax[j].set_xlabel('Number of tracks'); ax[j].set_xscale('log')
    ax[0].set_ylim(0,1.05); ax[1].set_ylim(0,1.05)
fig.tight_layout(); _save(fig,'fig2_fidelity'); plt.show()

In [ ]:
# Fig 3 — solver timing vs T (recorded t_solve from the store)
fig, axx = plt.subplots(figsize=(7,5))
if len(m):
    g=m.groupby('n_trk'); tim=pd.DataFrame({'n_trk':sorted(m.n_trk.unique())})
    tim['tC']=g['t_solve_C'].mean().values; tim['tQ']=g['t_solve_Q'].mean().values
    axx.plot(tim.n_trk,tim.tC,'o--',color=cC,label='classical'); axx.plot(tim.n_trk,tim.tQ,'o-',color=cQ,label='quantum (sv, simulator)')
    axx.set_xscale('log'); axx.set_yscale('log')
axx.set_xlabel('Number of tracks'); axx.set_ylabel('wall time per solve [s]'); axx.set_title('Solver timing'); axx.legend()
fig.tight_layout(); _save(fig,'fig3_timing'); plt.show()

In [ ]:
# Fig 4 — §14e mirror: efficiency & false rate, classical vs quantum (statevector)
fig, ax = plt.subplots(1,2, figsize=(13,5))
_err(ax[0],agg,'eff_C','D--',cC,'classical',mfc='white'); _err(ax[0],agg,'eff_Q','o-',cQ,'quantum (sv)')
ax[0].axhline(1,color='gray',ls='--',lw=1); ax[0].set_ylim(0,1.05); ax[0].set_title('§14e mirror — efficiency'); ax[0].legend(loc='lower left')
_err(ax[1],agg,'far_C','D--',cC,'classical',mfc='white'); _err(ax[1],agg,'far_Q','o-',cQ,'quantum (sv)')
ax[1].set_yscale('log'); ax[1].set_title('§14e mirror — false rate'); ax[1].legend(loc='upper left')
for a in ax: a.set_xlabel('Number of tracks'); a.set_xscale('log')
fig.tight_layout(); _save(fig,'fig4_seg14e_mirror'); plt.show()

In [ ]:
# Fig 5 — threshold fix: quantum absolute tau=0.35 vs relative tau*max (from raw vectors)
tau=V.threshold(); rows=[]
for r in recs:
    sQ=r.get('sol_Q_scaled')
    if sQ is None: continue
    tr=r['truth']; ntc=max(int(tr.sum()),1)
    aA=sQ>tau; qm=float(sQ.max()); aR=(sQ>tau*qm) if qm>0 else np.zeros_like(sQ,bool)
    rows.append(dict(n_trk=r['n_trk'],
        eff_abs=(aA&tr).sum()/ntc, far_abs=(aA&~tr).sum()/max(int(aA.sum()),1),
        eff_rel=(aR&tr).sum()/ntc, far_rel=(aR&~tr).sum()/max(int(aR.sum()),1)))
fig, ax = plt.subplots(1,2, figsize=(13,5))
if rows:
    g=pd.DataFrame(rows).groupby('n_trk').mean().reset_index()
    ax[0].plot(g.n_trk,g.eff_abs,'o-',color=cQ,label='absolute tau=0.35'); ax[0].plot(g.n_trk,g.eff_rel,'s--',color=cF,label='relative tau*max')
    ax[0].set_ylim(0,1.05); ax[0].set_title('quantum efficiency: abs vs rel threshold'); ax[0].legend()
    ax[1].plot(g.n_trk,g.far_abs,'o-',color=cQ,label='absolute'); ax[1].plot(g.n_trk,g.far_rel,'s--',color=cF,label='relative')
    ax[1].set_yscale('log'); ax[1].set_title('quantum false rate: abs vs rel threshold'); ax[1].legend()
    for a in ax: a.set_xlabel('Number of tracks'); a.set_xscale('log')
else: print('no quantum vectors yet')
fig.tight_layout(); _save(fig,'fig5_threshold_abs_vs_rel'); plt.show()

In [ ]:
# Fig 6 — false-segment leakage probability and threshold-free AUC (from raw vectors)
def _auc(s,tr):
    s=np.asarray(s,float); tr=np.asarray(tr,bool); n1=int(tr.sum()); n0=int((~tr).sum())
    if not n1 or not n0: return np.nan
    rank=np.argsort(np.argsort(s))+1
    return (rank[tr].sum()-n1*(n1+1)/2)/(n1*n0)
tau=V.threshold(); rows=[]
for r in recs:
    sQ=r.get('sol_Q_scaled')
    if sQ is None: continue
    tr=r['truth']; nf=max(int((~tr).sum()),1)
    rows.append(dict(n_trk=r['n_trk'], p_leak=(sQ[~tr]>tau).sum()/nf, auc=_auc(sQ,tr)))
fig, ax = plt.subplots(1,2, figsize=(13,5))
if rows:
    g=pd.DataFrame(rows).groupby('n_trk').mean().reset_index()
    ax[0].plot(g.n_trk,g.p_leak,'o-',color=cF); ax[0].set_yscale('log')
    ax[0].set_title('false-segment leak prob = N_false_act / N_false_all'); ax[0].set_xlabel('Number of tracks'); ax[0].set_xscale('log')
    ax[1].plot(g.n_trk,g.auc,'o-',color=cQ); ax[1].set_ylim(0.4,1.02)
    ax[1].set_title('threshold-free AUC  P(s_Q[true] > s_Q[false])'); ax[1].set_xlabel('Number of tracks'); ax[1].set_xscale('log')
else: print('no quantum vectors yet')
fig.tight_layout(); _save(fig,'fig6_leakage_auc'); plt.show()

## Notes / provenance

- Source of truth: `/data/bfys/gscriven/qtrk_store` (events regenerated on demand; solutions content-addressed; metrics recomputed via `qtrk_view`/`qtrk_pipeline.load_metrics`).
- Threshold is gamma-aware absolute (0.35 at gamma=3); false rate = `N_false_active / N_active`.
- **Omitted vs the old notebook (statevector-only store):** downstream tracking (Fig 3 old), and the statevector-vs-sampling / hires-sampling comparison suite (Sec 7e-7q). Re-enable sampling in the Condor campaign to restore those.
- The fixed-epsilon (`*_eps2`) sweeps you queued will appear here automatically once solved + `build_metrics.py` is re-run; this notebook just re-reads the view.